Importing Libraries

In [25]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler , TargetEncoder , LabelEncoder
from imblearn.over_sampling import SMOTE
import os
import joblib
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_parquet('../data/processed/engineered_features.parquet')
df

,isFraud,TransactionAmt,ProductCD,addr1,addr2,P_emaildomain,C1,C2,C3,C4,...,V317,V318,V319,V320,V321,hour_sin,hour_cos,card_id,amt_log,amt_zscore_card
0,0,68.50,W,315.0,87.0,unknown,1.0,1.0,0.0,0.0,...,117.0,0.0,0.000000,0.000000,0.000000,0.000000,1.000000,13926_-999.0_150.0_discover_142.0_credit,4.241327,-0.898847
1,0,29.00,W,325.0,87.0,gmail.com,1.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.000073,1.000000,2755_404.0_150.0_mastercard_102.0_credit,3.401197,-0.513125
2,0,59.00,W,330.0,87.0,outlook.com,1.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.005018,0.999987,4663_490.0_150.0_visa_166.0_debit,4.094345,-0.378888
3,0,50.00,W,476.0,87.0,yahoo.com,2.0,5.0,0.0,0.0,...,1404.0,790.0,0.000000,0.000000,0.000000,0.007199,0.999974,18132_567.0_150.0_mastercard_117.0_debit,3.931826,-0.383042
4,0,50.00,H,420.0,87.0,gmail.com,1.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,0.007708,0.999970,4497_514.0_150.0_mastercard_102.0_credit,3.931826,-0.829466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0,49.00,W,272.0,87.0,unknown,2.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,-0.011126,0.999938,6550_-999.0_150.0_visa_226.0_debit,3.912023,-0.356113
590536,0,39.50,W,204.0,87.0,gmail.com,1.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,-0.010981,0.999940,10444_225.0_150.0_mastercard_224.0_debit,3.701302,-2.064279
590537,0,30.95,W,231.0,87.0,gmail.com,1.0,1.0,0.0,0.0,...,0.0,0.0,0.000000,0.000000,0.000000,-0.008799,0.999961,12037_595.0_150.0_mastercard_224.0_debit,3.464172,-0.585652
590538,0,117.00,W,387.0,87.0,aol.com,1.0,1.0,0.0,0.0,...,2234.0,0.0,0.000000,0.000000,0.000000,-0.008145,0.999967,7826_481.0_150.0_mastercard_224.0_debit,4.770685,-0.032260


Rechecking everything

In [3]:
df.shape

(590540, 217)

In [4]:
df.isnull().sum().sum()

np.int64(0)

In [5]:
df['isFraud'].mean()*100

np.float64(3.4990009144173126)

In [6]:
#Ensuring fraud rate increases over time

print(f"First 10k rows fraud rate:  {df.head(10000)['isFraud'].mean()*100:.2f}%")
print(f"Last 10k rows fraud rate:   {df.tail(10000)['isFraud'].mean()*100:.2f}%")

First 10k rows fraud rate:  2.65%
Last 10k rows fraud rate:   3.99%


Temporal split (since data is time ordered , we cant random split)

In [7]:
# 75/25 train/test split : first 75% to training and last 25% to testing
split_index = int(len(df)*0.75)
split_index

442905

In [8]:
train_df = df.iloc[ : split_index].copy()
test_df = df.iloc[split_index : ].copy()

In [9]:
print(f"Shape of training set: {len(train_df)}")
print(f"Shape of test set: {len(test_df)}")
print(f"Train fraud rate: {train_df['isFraud'].mean()*100:.2f}%")
print(f"Test fraud rate:  {test_df['isFraud'].mean()*100:.2f}%")

Shape of training set: 442905
Shape of test set: 147635
Train fraud rate: 3.51%
Test fraud rate:  3.45%


Validation splitting from training set 

In [10]:
# 80/20 training/validation split : first 80% to training and last 20% to validation
val_index = int(len(train_df)*0.80)
val_index

354324

In [11]:
train_val_df = train_df.iloc[ : val_index].copy()
val_df = train_df.iloc[val_index : ].copy()

In [12]:
print(f"Shape of training validation set: {len(train_val_df)}")
print(f"Shape of validation set: {len(val_df)}")
print(f"Training validation fraud rate: {train_val_df['isFraud'].mean()*100:.2f}%")
print(f"Validation fraud rate:  {val_df['isFraud'].mean()*100:.2f}%")

Shape of training validation set: 354324
Shape of validation set: 88581
Training validation fraud rate: 3.38%
Validation fraud rate:  4.04%


Seperating training , validation and test set 

In [13]:
X_train = train_val_df.drop(columns = ['isFraud'])
y_train = train_val_df['isFraud']

X_val = val_df.drop(columns = ['isFraud'])
y_val = val_df['isFraud']

X_test = test_df.drop(columns = ['isFraud'])
y_test = test_df['isFraud']

In [14]:
print(f"X_train: {X_train.shape}, y_train fraud rate: {y_train.mean()*100:.2f}%")
print(f"X_val:   {X_val.shape},   y_val fraud rate:   {y_val.mean()*100:.2f}%")
print(f"X_test:  {X_test.shape},  y_test fraud rate:  {y_test.mean()*100:.2f}%")

X_train: (354324, 216), y_train fraud rate: 3.38%
X_val:   (88581, 216),   y_val fraud rate:   4.04%
X_test:  (147635, 216),  y_test fraud rate:  3.45%


Target Encoding 3 columns

In [15]:
te = TargetEncoder(target_type = 'binary' , smooth = 'auto')
te_cols = ['P_emaildomain' , 'ProductCD' , 'card_id']
X_train[te_cols] = te.fit_transform(X_train[te_cols] , y_train)
X_val[te_cols] = te.transform(X_val[te_cols])
X_test[te_cols] = te.transform(X_test[te_cols])

In [16]:
fraud_idx = y_train[y_train==1].index
print(f"P_emaildomain_te fraud mean: {X_train.loc[fraud_idx, 'P_emaildomain'].mean():.4f}")
print(f"card_id_te fraud mean:       {X_train.loc[fraud_idx, 'card_id'].mean():.4f}")
print(f"ProductCD_te fraud mean:     {X_train.loc[fraud_idx, 'ProductCD'].mean():.4f}")

P_emaildomain_te fraud mean: 0.0396
card_id_te fraud mean:       0.1355
ProductCD_te fraud mean:     0.0567


In [17]:
joblib.dump(te , "../models/target_encoder.pkl")

['../models/target_encoder.pkl']

Label Encoding remaining columns

In [18]:
X_train.select_dtypes(include='object').columns

Index(['M1', 'M2', 'M3', 'M4', 'M6'], dtype='object')

In [19]:
cat_cols_M = X_train.select_dtypes(include = object).columns.tolist()
le_encoders = {}
for col in cat_cols_M:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col].astype(str))
    X_val[col]   = le.transform(X_val[col].astype(str))
    X_test[col]  = le.transform(X_test[col].astype(str))

    le_encoders[col] = le
    print(f"Encoded column : {col}")

Encoded column : M1
Encoded column : M2
Encoded column : M3
Encoded column : M4
Encoded column : M6


In [20]:
X_train.select_dtypes(include = object).columns

Index([], dtype='object')

In [21]:
joblib.dump(le_encoders, '../models/label_encoders.pkl')

['../models/label_encoders.pkl']

Standard Scaling 

In [22]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train) , 
                              columns = X_train.columns , 
                              index = X_train.index)

X_val_scaled = pd.DataFrame(scaler.transform(X_val) , 
                              columns = X_val.columns , 
                              index = X_val.index)

X_test_scaled = pd.DataFrame(scaler.transform(X_test) , 
                              columns = X_test.columns , 
                              index = X_test.index)

In [23]:
print(f" Mean of scaled dataset : {X_train_scaled.mean().mean():.6f}")
print(f" STD of scaled dataset : {X_train_scaled.std().mean():.6f}")

 Mean of scaled dataset : -0.000000
 STD of scaled dataset : 0.995372


In [24]:
joblib.dump(scaler, '../models/Standard_scaler.pkl')

['../models/Standard_scaler.pkl']

SMOTE on unscaled dataset (for XGBoost and LighGBM)

In [26]:
print("Before Applying Smote")
print(f"Shape of training set : {X_train.shape}")
print(f"\nNumber of fraud cases : {y_train.sum()}")
print(f"\nPercentage of fraud cases : {y_train.mean()*100:.3f}")

Before Applying Smote
Shape of training set : (354324, 216)

Number of fraud cases : 11988

Percentage of fraud cases : 3.383


In [27]:
smote = SMOTE(random_state = 42 , k_neighbors = 5)
X_train_smote , y_train_smote = smote.fit_resample(X_train , y_train)

In [28]:
print("After Applying Smote")
print(f"Shape of training set : {X_train_smote.shape}")
print(f"\nNumber of fraud cases : {y_train_smote.sum()}")
print(f"\nPercentage of fraud cases : {y_train_smote.mean()*100:.3f}")

After Applying Smote
Shape of training set : (684672, 216)

Number of fraud cases : 342336

Percentage of fraud cases : 50.000


SMOTE on scaled dataset (for Linear Regression and AutoEncoder)

In [29]:
print("Before Applying Smote")
print(f"Shape of training set : {X_train_scaled.shape}")
print(f"\nNumber of fraud cases : {y_train.sum()}")
print(f"\nPercentage of fraud cases : {y_train.mean()*100:.3f}")

Before Applying Smote
Shape of training set : (354324, 216)

Number of fraud cases : 11988

Percentage of fraud cases : 3.383


In [30]:
X_train_scaled_smote , y_train_scaled_smote = smote.fit_resample(X_train_scaled , y_train)

In [31]:
print("After Applying Smote")
print(f"Shape of training set : {X_train_scaled_smote.shape}")
print(f"\nNumber of fraud cases : {y_train_scaled_smote.sum()}")
print(f"\nPercentage of fraud cases : {y_train_scaled_smote.mean()*100:.3f}")

After Applying Smote
Shape of training set : (684672, 216)

Number of fraud cases : 342336

Percentage of fraud cases : 50.000


In [32]:
X_train_smote

,TransactionAmt,ProductCD,addr1,addr2,P_emaildomain,C1,C2,C3,C4,C5,...,V317,V318,V319,V320,V321,hour_sin,hour_cos,card_id,amt_log,amt_zscore_card
0,68.500000,0.020312,315.000000,87.0,0.028714,1.000000,1.000000,0.0,0.000000,0.0,...,117.000000,0.000000,0.0,0.0,0.0,0.000000,1.000000,0.000000,4.241327,-0.898847
1,29.000000,0.020569,325.000000,87.0,0.042128,1.000000,1.000000,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.000073,1.000000,0.072204,3.401197,-0.513125
2,59.000000,0.020405,330.000000,87.0,0.085979,1.000000,1.000000,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.005018,0.999987,0.016745,4.094345,-0.378888
3,50.000000,0.020312,476.000000,87.0,0.022902,2.000000,5.000000,0.0,0.000000,0.0,...,1404.000000,790.000000,0.0,0.0,0.0,0.007199,0.999974,0.012280,3.931826,-0.383042
4,50.000000,0.042736,420.000000,87.0,0.042175,1.000000,1.000000,0.0,0.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,0.007708,0.999970,0.125540,3.931826,-0.829466
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684667,32.404202,0.106878,-999.000000,-999.0,0.042078,1.000000,1.000000,0.0,1.000000,0.0,...,0.000000,0.000000,0.0,0.0,0.0,-0.106951,-0.436653,0.134310,3.508574,-0.302702
684668,609.009896,0.020343,276.404519,87.0,0.028660,2.674937,2.337469,0.0,0.000000,0.0,...,960.794267,144.431841,0.0,0.0,0.0,-0.376155,-0.511481,0.109202,6.411461,1.800123
684669,18.360221,0.058544,327.400158,87.0,0.028568,7.280348,3.960063,0.0,3.920126,0.0,...,5.199684,0.000000,0.0,0.0,0.0,-0.862451,-0.104862,0.033475,2.946795,-0.616030
684670,144.839982,0.020312,351.279521,87.0,0.042258,5.165875,4.624406,0.0,0.000000,0.0,...,604.762902,22.468028,0.0,0.0,0.0,-0.754481,0.451706,0.097055,4.731226,0.237901


In [33]:
X_train_scaled_smote

,TransactionAmt,ProductCD,addr1,addr2,P_emaildomain,C1,C2,C3,C4,C5,...,V317,V318,V319,V320,V321,hour_sin,hour_cos,card_id,amt_log,amt_zscore_card
0,-0.298117,-0.486258,0.400157,0.352005,-0.365460,-0.096573,-0.092071,-0.039979,-0.068078,-0.212086,...,0.009530,-0.099390,-0.057307,-0.090320,-0.070350,0.518512,1.113571,-0.538691,-0.154811,-0.917744
1,-0.478791,-0.477015,0.424273,0.352005,0.594692,-0.096573,-0.092071,-0.039979,-0.068078,-0.212086,...,-0.074502,-0.099390,-0.057307,-0.090320,-0.070350,0.518627,1.113571,0.660032,-1.056909,-0.521134
2,-0.341570,-0.482910,0.436331,0.352005,3.733375,-0.096573,-0.092071,-0.039979,-0.068078,-0.212086,...,-0.074502,-0.099390,-0.057307,-0.090320,-0.070350,0.526404,1.113551,-0.260698,-0.312634,-0.383109
3,-0.382737,-0.486258,0.788425,0.352005,-0.781453,-0.090574,-0.071410,-0.039979,-0.068078,-0.212086,...,0.933883,1.504848,-0.057307,-0.090320,-0.070350,0.529835,1.113530,-0.334814,-0.487141,-0.387380
4,-0.382737,0.320212,0.653376,0.352005,0.598060,-0.096573,-0.092071,-0.039979,-0.068078,-0.212086,...,-0.074502,-0.099390,-0.057307,-0.090320,-0.070350,0.530635,1.113524,1.545511,-0.487141,-0.846404
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684667,-0.443260,2.639564,-2.768692,-2.846352,0.596419,-0.090418,-0.089421,-0.039979,-0.056217,-0.212086,...,-0.074502,-0.099390,-0.057307,-0.090320,-0.070350,1.904997,-1.150958,2.401209,-0.818224,-0.310933
684668,2.174192,-0.485128,0.307080,0.352005,-0.369272,-0.086525,-0.085163,-0.039979,-0.068078,-0.212086,...,0.615563,0.193905,-0.057307,-0.090320,-0.070350,-0.073053,-1.279264,1.274280,2.175394,1.857404
684669,-0.515566,0.938586,0.436331,0.352005,-0.363895,-0.055776,-0.079467,-0.039979,-0.027749,-0.212086,...,-0.074502,-0.099390,-0.057307,-0.090320,-0.070350,-0.935795,0.131152,0.474842,-1.393002,-0.682803
684670,0.422293,-0.482534,0.592700,0.352005,0.607152,-0.074332,-0.075718,-0.039979,-0.068078,-0.212086,...,0.613270,-0.099390,-0.057307,0.406219,0.464643,-0.980298,-0.082156,-0.133880,1.116118,0.538019


In [34]:
print(type(X_train_smote))
print(type(X_train_scaled_smote))

<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [35]:
print(type(y_train_smote))
print(type(y_train_scaled_smote))

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


In [37]:
X_train_smote.to_parquet('../data/processed/X_train.parquet', index=False)
pd.DataFrame(y_train_smote, columns=['isFraud']).to_parquet('../data/processed/y_train.parquet', index=False)

X_val.to_parquet('../data/processed/X_val.parquet', index=False)
pd.DataFrame(y_val, columns=['isFraud']).to_parquet('../data/processed/y_val.parquet', index=False)

X_test.to_parquet('../data/processed/X_test.parquet', index=False)
pd.DataFrame(y_test, columns=['isFraud']).to_parquet('../data/processed/y_test.parquet', index=False)

X_train_scaled_smote.to_parquet('../data/processed/X_train_scaled.parquet', index=False)
pd.DataFrame(y_train_scaled_smote, columns=['isFraud']).to_parquet('../data/processed/y_train_scaled.parquet', index=False)

X_val_scaled.to_parquet('../data/processed/X_val_scaled.parquet', index=False)
X_test_scaled.to_parquet('../data/processed/X_test_scaled.parquet', index=False)

print("All splits saved to data/processed/")

All splits saved to data/processed/
